# Bureau Scrub Comparison — Jan 2026 vs Nov 2025

**Purpose:** Compare two Experian bureau scrubs to understand:
- How credit scores shifted over ~2 months for the same users
- Which bureau parameters drove score improvements vs declines
- Patterns across improver / decliner / stable segments

**Output:** Insights feed into `knowledge_base/07_score_patterns.md` and the fine-tuning Q&A dataset.

---

**Files used (raw bureau RPT files):**
- Latest: `2026-01-28/RPT_*.txt`
- Latest-1: `2025-11-30/RPT_*.csv`

**Join key:** Phone number extracted from `RPT_PHONE` (CUSTOMER_ID in Nov 2025 was a reference key, not phone — phone is the reliable identifier across both scrubs)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 100})
sns.set_theme(style='whitegrid', palette='muted')
BLUE, RED, GREEN, ORANGE = '#2563eb', '#dc2626', '#16a34a', '#d97706'

# ── Scrub dates ───────────────────────────────────────────────────────────────
LATEST_DATE   = date(2026, 1, 28)
LATEST_1_DATE = date(2025, 11, 30)

# ── File paths ────────────────────────────────────────────────────────────────
BASE = '/Users/ayushigupta/Documents/bureau_scrub/scrub files'

JAN26 = {
    'score':  f'{BASE}/2026-01-28/RPT_SCOREV3.txt',
    'phone':  f'{BASE}/2026-01-28/RPT_PHONE.txt',
    'ar':     f'{BASE}/2026-01-28/RPT_AR.txt',
    'enq':    f'{BASE}/2026-01-28/RPT_ENQ.txt',
    'dob':    f'{BASE}/2026-01-28/RPT_NAME_DOB.txt',
}

NOV25 = {
    'score':  f'{BASE}/2025-11-30/RPT_V3_SCORE.csv',
    'phone':  f'{BASE}/2025-11-30/RPT_PHONE.txt',
    'ar':     f'{BASE}/2025-11-30/RPT_AR.csv',
    'enq':    f'{BASE}/2025-11-30/RPT_ENQ.csv',
}

# Columns needed from RPT_AR (skip the wide monthly history columns)
AR_COLS = [
    'CUSTOMER_ID', 'M_SUB_ID', 'ACCT_TYPE_CD', 'OPEN_DT', 'CLOSED_DT',
    'ASSET_CLASS_CD', 'BALANCE_AM', 'CREDIT_LIMIT_AM', 'ORIG_LOAN_AM',
    'DAYS_PAST_DUE', 'PAYMENT_HISTORY_GRID', 'WRITTEN_OFF_AND_SETTLED_STATUS',
    'CHARGE_OFF_AM', 'RESPONSIBILITY_CD'
]

# Secured account type codes (Home Loan, Auto Loan, Gold Loan, LAP, etc.)
SECURED_TYPES   = {'03', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '35', '36'}
# Unsecured (Personal Loan, Business Loan, Credit Card, OD, etc.)
UNSECURED_TYPES = {'06', '10', '11', '12', '13', '14', '15', '16', '17', '29', '30', '31', '32'}
CC_TYPE = '10'

# NPA asset class codes
NPA_CLASSES = {'SUB', 'DBT', 'LSS'}

print('Config loaded.')
print(f'Latest scrub:   {LATEST_DATE}')
print(f'Latest-1 scrub: {LATEST_1_DATE}')

## Section 1 — Load Scores & Build Phone Mappings

**Why phone as the join key?**  
The Nov 2025 scrub was submitted with a reference key (not phone), so `CUSTOMER_ID` in those RPT files is an internal reference string. `RPT_PHONE` maps each CUSTOMER_ID → actual phone number. Since Jan 2026 CUSTOMER_IDs are phone numbers directly, phone is the clean universal key.

The `PHONE` column in RPT_PHONE has a `91` country code prefix — we strip that to get the 10-digit mobile number.

In [ ]:
def load_phone_mapping(path):
    """
    Reads RPT_PHONE and returns a dict: CUSTOMER_ID -> phone_10digit.
    Strips the 91 country code prefix from the PHONE column.
    Keeps only the most recent phone number per customer.
    """
    df = pd.read_csv(path, sep='|', usecols=['CUSTOMER_ID', 'PHONE', 'DATE_REPORTED'],
                     dtype=str, low_memory=False)
    df['CUSTOMER_ID'] = df['CUSTOMER_ID'].str.strip()
    df['PHONE'] = df['PHONE'].str.strip()
    # Strip 91 prefix → 10-digit phone
    df['phone'] = df['PHONE'].str.replace(r'^91', '', regex=True).str.strip()
    df = df[df['phone'].str.len() == 10]  # keep only valid 10-digit
    # Keep one phone per customer (latest reported)
    df = df.sort_values('DATE_REPORTED').drop_duplicates('CUSTOMER_ID', keep='last')
    return dict(zip(df['CUSTOMER_ID'], df['phone']))


print('Loading phone mappings...')
phone_map_jan26 = load_phone_mapping(JAN26['phone'])
phone_map_nov25 = load_phone_mapping(NOV25['phone'])

print(f'Jan 2026: {len(phone_map_jan26):,} customer → phone mappings')
print(f'Nov 2025: {len(phone_map_nov25):,} customer → phone mappings')

In [ ]:
def load_scores(path, phone_map):
    """
    Loads RPT_SCORE file and maps CUSTOMER_ID → phone.
    Returns DataFrame with columns: phone, score.
    """
    df = pd.read_csv(path, sep='|', dtype={'CUSTOMER_ID': str, 'SCORE_V3': str},
                     low_memory=False)
    df.columns = df.columns.str.strip()
    df['CUSTOMER_ID'] = df['CUSTOMER_ID'].str.strip()
    df['score'] = pd.to_numeric(df['SCORE_V3'], errors='coerce')
    df['phone'] = df['CUSTOMER_ID'].map(phone_map)
    df = df[df['phone'].notna() & df['score'].notna()]
    # Drop duplicates (same phone in multiple scrub batches — keep highest score row)
    df = df.sort_values('score', ascending=False).drop_duplicates('phone', keep='first')
    return df[['phone', 'score']].reset_index(drop=True)


print('Loading scores...')
scores_jan26 = load_scores(JAN26['score'], phone_map_jan26)
scores_nov25 = load_scores(NOV25['score'], phone_map_nov25)

print(f'Jan 2026 scores: {len(scores_jan26):,} customers with valid phone + score')
print(f'Nov 2025 scores: {len(scores_nov25):,} customers with valid phone + score')
print(f'\nJan 2026 score stats:')
print(scores_jan26['score'].describe())
print(f'\nNov 2025 score stats:')
print(scores_nov25['score'].describe())

In [ ]:
# ── Join on phone — only customers present in BOTH scrubs ─────────────────────
df = pd.merge(
    scores_jan26.rename(columns={'score': 'score_jan26'}),
    scores_nov25.rename(columns={'score': 'score_nov25'}),
    on='phone', how='inner'
)
df['score_delta'] = df['score_jan26'] - df['score_nov25']

# Segment
def segment(delta):
    if delta >= 25:   return 'Improver'
    if delta <= -25:  return 'Decliner'
    return 'Stable'

df['segment'] = df['score_delta'].apply(segment)

print(f'Matched customers (in both scrubs): {len(df):,}')
print(f'\nSegment breakdown:')
print(df['segment'].value_counts())
print(f'\nScore delta stats:')
print(df['score_delta'].describe())

## Section 2 — Score Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Score distribution comparison
ax = axes[0]
ax.hist(df['score_nov25'].dropna(), bins=40, alpha=0.6, color=ORANGE, label='Nov 2025')
ax.hist(df['score_jan26'].dropna(), bins=40, alpha=0.6, color=BLUE, label='Jan 2026')
ax.set_title('Score Distribution — Both Scrubs')
ax.set_xlabel('Score')
ax.set_ylabel('Customers')
ax.legend()

# Score delta distribution
ax = axes[1]
ax.hist(df['score_delta'].dropna(), bins=50, color=GREEN, edgecolor='white')
ax.axvline(0, color='black', linestyle='--', alpha=0.7, label='No change')
ax.axvline(df['score_delta'].mean(), color=RED, linestyle='--', alpha=0.7,
           label=f"Mean = {df['score_delta'].mean():.1f}")
ax.set_title('Score Delta (Jan 2026 − Nov 2025)')
ax.set_xlabel('Delta')
ax.set_ylabel('Customers')
ax.legend()

# Segment breakdown
ax = axes[2]
seg_counts = df['segment'].value_counts()
colors = [GREEN if s == 'Improver' else RED if s == 'Decliner' else BLUE
          for s in seg_counts.index]
bars = ax.bar(seg_counts.index, seg_counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, seg_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Customer Segments')
ax.set_ylabel('Customers')

plt.tight_layout()
plt.savefig('../notebooks/score_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nMedian score change by segment:')
print(df.groupby('segment')['score_delta'].median())

In [ ]:
# Score band migration — where did each Nov 2025 band move to in Jan 2026?
def score_band(s):
    if pd.isna(s): return 'Unknown'
    if s >= 750:   return '750+ (Prime)'
    if s >= 700:   return '700-749 (Good)'
    if s >= 650:   return '650-699 (Fair)'
    if s >= 600:   return '600-649 (Poor)'
    return '<600 (Very Poor)'

df['band_nov25'] = df['score_nov25'].apply(score_band)
df['band_jan26'] = df['score_jan26'].apply(score_band)

band_order = ['750+ (Prime)', '700-749 (Good)', '650-699 (Fair)',
              '600-649 (Poor)', '<600 (Very Poor)']

migration = pd.crosstab(df['band_nov25'], df['band_jan26'],
                         normalize='index').reindex(index=band_order,
                                                    columns=band_order, fill_value=0)

plt.figure(figsize=(10, 6))
sns.heatmap(migration * 100, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': '% of row'})
plt.title('Score Band Migration (Nov 2025 → Jan 2026)\n% of customers in each Nov band who moved to each Jan band')
plt.xlabel('Jan 2026 Band')
plt.ylabel('Nov 2025 Band')
plt.tight_layout()
plt.savefig('../notebooks/score_band_migration.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 3 — Feature Engineering from RPT_AR

**What RPT_AR contains:** One row per tradeline (loan/credit card account) per customer. A customer with 5 accounts has 5 rows.

**Key fields used:**
- `PAYMENT_HISTORY_GRID` — 36-char string, each char = one month's payment status (most recent first). `0`=on time, `1`=30 DPD, `2`=60 DPD, `3`=90+ DPD, `X`=account not open, `?`=no data
- `ASSET_CLASS_CD` — current NPA status: STD/SMA/SUB/DBT/LSS
- `WRITTEN_OFF_AND_SETTLED_STATUS` — write-off or settlement marker
- `RESPONSIBILITY_CD` — `I`=individual (we include), `J`=joint, `G`=guarantor, `C`=co-applicant (we exclude)

**Memory note:** AR files are 3.3GB and 1.2GB. We use `usecols` to load only the 14 columns we need, processed in 50K-row chunks.

In [ ]:
DPD_30_CHARS = set('123456789')
DPD_60_CHARS = set('23456789')
DPD_90_CHARS = set('3456789')
WRITEOFF_CODES = {'W', 'L', 'S', 'WO', 'SO', 'W-O', 'W/O'}  # written off / settled


def dpd_flag(grid, chars, window):
    """Check if any DPD event of given severity exists within the window months."""
    if not isinstance(grid, str) or len(grid) == 0:
        return 0
    return int(any(c in chars for c in grid[:window] if c not in ('?', 'X', '-', ' ')))


def process_ar_chunk(chunk, phone_map):
    """Process one chunk of RPT_AR and return per-customer feature rows."""
    chunk.columns = chunk.columns.str.strip()
    chunk['CUSTOMER_ID'] = chunk['CUSTOMER_ID'].astype(str).str.strip()
    chunk['phone'] = chunk['CUSTOMER_ID'].map(phone_map)
    chunk = chunk[chunk['phone'].notna()]
    if chunk.empty:
        return pd.DataFrame()

    # Only individual accounts (exclude guarantor/co-applicant)
    chunk = chunk[chunk['RESPONSIBILITY_CD'].astype(str).str.strip().isin(['I', 'S', '1', ''])]

    # Numeric conversions
    for col in ['BALANCE_AM', 'CREDIT_LIMIT_AM', 'ORIG_LOAN_AM', 'CHARGE_OFF_AM', 'DAYS_PAST_DUE']:
        chunk[col] = pd.to_numeric(chunk[col], errors='coerce').fillna(0)

    # Active = CLOSED_DT is null/empty
    chunk['is_active'] = chunk['CLOSED_DT'].astype(str).str.strip().isin(['', 'nan', 'NaN', 'None'])

    # Account type
    chunk['ACCT_TYPE_CD'] = chunk['ACCT_TYPE_CD'].astype(str).str.strip().str.zfill(2)

    # DPD flags from payment history grid
    chunk['dpd30_12m'] = chunk['PAYMENT_HISTORY_GRID'].apply(dpd_flag, chars=DPD_30_CHARS, window=12)
    chunk['dpd60_24m'] = chunk['PAYMENT_HISTORY_GRID'].apply(dpd_flag, chars=DPD_60_CHARS, window=24)
    chunk['dpd90_36m'] = chunk['PAYMENT_HISTORY_GRID'].apply(dpd_flag, chars=DPD_90_CHARS, window=36)

    # NPA / negative flag
    chunk['is_npa'] = chunk['ASSET_CLASS_CD'].astype(str).str.strip().str.upper().isin(NPA_CLASSES).astype(int)

    # Write-off / settled
    chunk['is_writeoff'] = chunk['WRITTEN_OFF_AND_SETTLED_STATUS'].astype(str).str.strip().str.upper().isin(
        WRITEOFF_CODES).astype(int)

    # Is credit card
    chunk['is_cc'] = (chunk['ACCT_TYPE_CD'] == CC_TYPE).astype(int)

    # Aggregate per phone
    agg = chunk.groupby('phone').agg(
        total_accounts    = ('CUSTOMER_ID', 'count'),
        active_accounts   = ('is_active', 'sum'),
        total_outstanding = ('BALANCE_AM', 'sum'),
        total_orig_loan   = ('ORIG_LOAN_AM', 'sum'),
        has_dpd30_12m     = ('dpd30_12m', 'max'),
        num_dpd30_12m     = ('dpd30_12m', 'sum'),
        has_dpd60_24m     = ('dpd60_24m', 'max'),
        has_dpd90_36m     = ('dpd90_36m', 'max'),
        has_npa           = ('is_npa', 'max'),
        num_npa           = ('is_npa', 'sum'),
        has_writeoff      = ('is_writeoff', 'max'),
        total_writeoff_am = ('CHARGE_OFF_AM', 'sum'),
        cc_count_active   = ('is_cc', lambda x: x[chunk.loc[x.index, 'is_active']].sum()),
        cc_limit_total    = ('CREDIT_LIMIT_AM', lambda x: x[chunk.loc[x.index, 'is_cc'].astype(bool) &
                                                              chunk.loc[x.index, 'is_active']].sum()),
        cc_balance_total  = ('BALANCE_AM', lambda x: x[chunk.loc[x.index, 'is_cc'].astype(bool) &
                                                        chunk.loc[x.index, 'is_active']].sum()),
    ).reset_index()

    agg['cc_util_pct'] = np.where(
        agg['cc_limit_total'] > 0,
        (agg['cc_balance_total'] / agg['cc_limit_total'] * 100).round(1),
        np.nan
    )
    return agg


print('AR feature engineering functions ready.')

In [ ]:
def load_ar_features(path, phone_map, label=''):
    """
    Loads RPT_AR in 50K-row chunks, processes each, and combines results.
    Uses only AR_COLS to minimise memory usage on large (1-3GB) files.
    """
    print(f'Loading AR features for {label}...')
    all_chunks = []
    chunk_count = 0
    reader = pd.read_csv(path, sep='|', usecols=AR_COLS, dtype=str,
                         low_memory=False, chunksize=50_000)
    for chunk in reader:
        processed = process_ar_chunk(chunk, phone_map)
        if not processed.empty:
            all_chunks.append(processed)
        chunk_count += 1
        if chunk_count % 20 == 0:
            print(f'  ...processed {chunk_count * 50_000:,} rows')

    combined = pd.concat(all_chunks, ignore_index=True)
    # Re-aggregate in case same phone appeared across chunks
    final = combined.groupby('phone').agg(
        total_accounts    = ('total_accounts', 'sum'),
        active_accounts   = ('active_accounts', 'sum'),
        total_outstanding = ('total_outstanding', 'sum'),
        total_orig_loan   = ('total_orig_loan', 'sum'),
        has_dpd30_12m     = ('has_dpd30_12m', 'max'),
        num_dpd30_12m     = ('num_dpd30_12m', 'sum'),
        has_dpd60_24m     = ('has_dpd60_24m', 'max'),
        has_dpd90_36m     = ('has_dpd90_36m', 'max'),
        has_npa           = ('has_npa', 'max'),
        num_npa           = ('num_npa', 'sum'),
        has_writeoff      = ('has_writeoff', 'max'),
        total_writeoff_am = ('total_writeoff_am', 'sum'),
        cc_count_active   = ('cc_count_active', 'sum'),
        cc_limit_total    = ('cc_limit_total', 'sum'),
        cc_balance_total  = ('cc_balance_total', 'sum'),
        cc_util_pct       = ('cc_util_pct', 'mean'),
    ).reset_index()

    print(f'Done. {len(final):,} customers with AR features.')
    return final


ar_jan26 = load_ar_features(JAN26['ar'], phone_map_jan26, 'Jan 2026')
ar_nov25 = load_ar_features(NOV25['ar'], phone_map_nov25, 'Nov 2025')

## Section 4 — Enquiry Analysis

**Why enquiries matter:** Each hard enquiry (loan application, credit card application) reduces a score by ~5-10 points. Multiple enquiries in a short window signal credit-hungry behaviour and can cause a sharper drop.

We compute enquiry counts per customer for the 6 months and 12 months before each scrub date.

In [ ]:
def load_enq_features(path, phone_map, scrub_date, label=''):
    """
    Loads RPT_ENQ and computes per-customer enquiry counts
    for 6 months and 12 months before the scrub date.
    """
    print(f'Loading enquiries for {label}...')
    df = pd.read_csv(path, sep='|', dtype=str, low_memory=False)
    df.columns = df.columns.str.strip()
    df['CUSTOMER_ID'] = df['CUSTOMER_ID'].astype(str).str.strip()
    df['phone'] = df['CUSTOMER_ID'].map(phone_map)
    df = df[df['phone'].notna()]

    df['INQ_DATE'] = pd.to_datetime(df['INQ_DATE'], format='%d/%m/%Y', errors='coerce')
    df = df[df['INQ_DATE'].notna()]

    cutoff_6m  = pd.Timestamp(scrub_date) - pd.DateOffset(months=6)
    cutoff_12m = pd.Timestamp(scrub_date) - pd.DateOffset(months=12)

    enq_6m  = df[df['INQ_DATE'] >= cutoff_6m].groupby('phone').size().rename('enq_6m')
    enq_12m = df[df['INQ_DATE'] >= cutoff_12m].groupby('phone').size().rename('enq_12m')

    result = pd.DataFrame(index=df['phone'].unique())
    result = result.join(enq_6m).join(enq_12m).fillna(0).astype(int).reset_index()
    result.columns = ['phone', 'enq_6m', 'enq_12m']

    print(f'Done. {len(result):,} customers with enquiry data.')
    return result


enq_jan26 = load_enq_features(JAN26['enq'], phone_map_jan26, LATEST_DATE,   'Jan 2026')
enq_nov25 = load_enq_features(NOV25['enq'], phone_map_nov25, LATEST_1_DATE, 'Nov 2025')

## Section 5 — Master Comparison Dataset

Join everything together for the matched customers (those in both scrubs). Then compute feature deltas — the *change* in each metric between the two scrubs. These deltas are what we'll correlate with score change.

In [ ]:
# Start from the matched score dataframe
master = df.copy()  # phone, score_jan26, score_nov25, score_delta, segment, band_*

# Join AR features
master = master.merge(ar_jan26.add_suffix('_j26').rename(columns={'phone_j26': 'phone'}),
                      on='phone', how='left')
master = master.merge(ar_nov25.add_suffix('_n25').rename(columns={'phone_n25': 'phone'}),
                      on='phone', how='left')

# Join ENQ features
master = master.merge(enq_jan26.add_suffix('_j26').rename(columns={'phone_j26': 'phone'}),
                      on='phone', how='left')
master = master.merge(enq_nov25.add_suffix('_n25').rename(columns={'phone_n25': 'phone'}),
                      on='phone', how='left')

# Compute deltas for key features
DELTA_FEATURES = [
    'total_accounts', 'active_accounts', 'total_outstanding',
    'has_dpd30_12m', 'num_dpd30_12m', 'has_dpd60_24m', 'has_dpd90_36m',
    'has_npa', 'has_writeoff', 'cc_util_pct', 'enq_6m', 'enq_12m'
]
for feat in DELTA_FEATURES:
    if f'{feat}_j26' in master.columns and f'{feat}_n25' in master.columns:
        master[f'delta_{feat}'] = master[f'{feat}_j26'] - master[f'{feat}_n25']

print(f'Master dataset: {len(master):,} customers, {len(master.columns)} columns')
master[['phone', 'score_nov25', 'score_jan26', 'score_delta', 'segment']].head()

## Section 6 — Segment Analysis

Compare Improvers, Decliners, and Stable customers across all key features.

In [ ]:
SEG_FEATURES = [
    ('has_dpd30_12m_j26', 'Has DPD 30+ (last 12m)'),
    ('has_dpd60_24m_j26', 'Has DPD 60+ (last 24m)'),
    ('has_dpd90_36m_j26', 'Has DPD 90+ (last 36m)'),
    ('has_npa_j26',       'Has NPA Account'),
    ('has_writeoff_j26',  'Has Write-off/Settlement'),
    ('enq_6m_j26',        'Avg Enquiries (6m)'),
    ('cc_util_pct_j26',   'Avg CC Utilisation %'),
    ('total_accounts_j26','Avg Total Accounts'),
    ('active_accounts_j26','Avg Active Accounts'),
]

seg_summary = {}
for col, label in SEG_FEATURES:
    if col in master.columns:
        seg_summary[label] = master.groupby('segment')[col].mean().round(2)

seg_df = pd.DataFrame(seg_summary).T
seg_df = seg_df[['Improver', 'Stable', 'Decliner']]
print('Segment profile (averages):')
print(seg_df.to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

plot_features = [
    ('has_dpd30_12m_j26', 'DPD 30+ rate (Jan 26)', '%'),
    ('has_npa_j26',       'NPA rate (Jan 26)', '%'),
    ('has_writeoff_j26',  'Write-off rate (Jan 26)', '%'),
    ('enq_6m_j26',        'Avg enquiries last 6m', 'count'),
    ('cc_util_pct_j26',   'Avg CC utilisation', '%'),
    ('active_accounts_j26','Avg active accounts', 'count'),
]

seg_order = ['Improver', 'Stable', 'Decliner']
seg_colors = [GREEN, BLUE, RED]

for ax, (col, title, unit) in zip(axes, plot_features):
    if col not in master.columns:
        continue
    vals = [master[master['segment'] == s][col].mean() for s in seg_order]
    if unit == '%':
        vals = [v * 100 if v <= 1 else v for v in vals]  # convert 0-1 flags to %
    bars = ax.bar(seg_order, vals, color=seg_colors, edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}{unit if unit=="%" else ""}', ha='center', fontsize=10)
    ax.set_title(title)
    ax.set_ylabel(unit)

plt.suptitle('Feature Comparison Across Segments (Jan 2026 snapshot)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/segment_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 7 — What Changed for Decliners vs Improvers?

Compare the *deltas* (what changed between Nov 2025 and Jan 2026) across segments.

In [ ]:
delta_cols = [c for c in master.columns if c.startswith('delta_')]
print('Change between scrubs — median delta by segment:')
delta_summary = master.groupby('segment')[delta_cols].median().round(3)
delta_summary.columns = [c.replace('delta_', '') for c in delta_cols]
print(delta_summary[['Improver', 'Stable', 'Decliner']].T.to_string())

In [ ]:
# Correlation between feature deltas and score delta
delta_corr_cols = [c for c in master.columns if c.startswith('delta_')]
corr = master[delta_corr_cols + ['score_delta']].corr()['score_delta'].drop('score_delta').sort_values()

plt.figure(figsize=(10, 6))
colors = [RED if v < 0 else GREEN for v in corr.values]
bars = plt.barh(corr.index.str.replace('delta_', ''), corr.values, color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Correlation with Score Delta (Jan 2026 − Nov 2025)\n(Green = more of this → higher score; Red = more → lower score)')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.savefig('../notebooks/feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# DPD rate by score band — how prevalent is DPD 30+ in each band?
dpd_by_band = master.groupby('band_jan26')['has_dpd30_12m_j26'].mean().mul(100).reindex([
    '750+ (Prime)', '700-749 (Good)', '650-699 (Fair)',
    '600-649 (Poor)', '<600 (Very Poor)'
]).dropna()

plt.figure(figsize=(10, 5))
bars = plt.bar(dpd_by_band.index, dpd_by_band.values,
               color=[GREEN, GREEN, ORANGE, RED, RED], edgecolor='white')
for bar, val in zip(bars, dpd_by_band.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=10)
plt.title('% of Customers with DPD 30+ in Last 12 Months — by Score Band (Jan 2026)')
plt.ylabel('% of customers')
plt.xlabel('Score Band')
plt.tight_layout()
plt.savefig('../notebooks/dpd_by_score_band.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 8 — Key Findings Summary

Numerical summary that feeds into `knowledge_base/07_score_patterns.md`.

In [ ]:
print('=' * 60)
print('KEY FINDINGS — Bureau Scrub Comparison')
print(f'Jan 2026 vs Nov 2025 | {len(master):,} matched customers')
print('=' * 60)

# Score movement
print(f"\n▶ Score Movement")
print(f"  Median score change:      {master['score_delta'].median():+.0f}")
print(f"  Mean score change:        {master['score_delta'].mean():+.1f}")
print(f"  Customers improved (≥25): {(master['segment']=='Improver').sum():,} ({(master['segment']=='Improver').mean()*100:.1f}%)")
print(f"  Customers declined (≤-25):{(master['segment']=='Decliner').sum():,} ({(master['segment']=='Decliner').mean()*100:.1f}%)")
print(f"  Customers stable:         {(master['segment']=='Stable').sum():,} ({(master['segment']=='Stable').mean()*100:.1f}%)")

# Score band distribution
print(f"\n▶ Score Band Distribution (Jan 2026)")
for band, cnt in master['band_jan26'].value_counts().items():
    pct = cnt / len(master) * 100
    print(f"  {band}: {cnt:,} ({pct:.1f}%)")

# DPD patterns
if 'has_dpd30_12m_j26' in master.columns:
    dpd30 = master['has_dpd30_12m_j26'].mean() * 100
    dpd90 = master['has_dpd90_36m_j26'].mean() * 100
    print(f"\n▶ DPD Patterns (Jan 2026)")
    print(f"  Customers with any DPD 30+ in last 12m: {dpd30:.1f}%")
    print(f"  Customers with any DPD 90+ in last 36m: {dpd90:.1f}%")

# NPA / write-off
if 'has_npa_j26' in master.columns:
    npa = master['has_npa_j26'].mean() * 100
    wo  = master['has_writeoff_j26'].mean() * 100
    print(f"\n▶ Credit Health (Jan 2026)")
    print(f"  Customers with NPA account:     {npa:.1f}%")
    print(f"  Customers with write-off/settlement: {wo:.1f}%")

# Enquiry patterns
if 'enq_6m_j26' in master.columns:
    print(f"\n▶ Enquiry Patterns (Jan 2026)")
    print(f"  Avg enquiries in last 6m:  {master['enq_6m_j26'].mean():.2f}")
    print(f"  Avg enquiries in last 12m: {master['enq_12m_j26'].mean():.2f}")
    hi_enq = (master['enq_6m_j26'] >= 3).mean() * 100
    print(f"  Customers with 3+ enquiries in 6m: {hi_enq:.1f}%")

print('\n' + '=' * 60)
print('Export master dataset to CSV for further use...')
master.to_csv('../notebooks/scrub_comparison_master.csv', index=False)
print('Saved: notebooks/scrub_comparison_master.csv')